Basado en el siguiente artículo:

jasonramchandani 
@rsamasiri rsamasiri Raksina Samasiri

#https://github.com/Refinitiv-API-Samples/Article.EikonAPI.Python.NewsSentimentAnalysis

In [0]:
#!pip install statsmodels
#!pip install linearmodels
#!pip install lseg.data
#!pip install pandas_datareader
#!pip install torch
#!pip3 install textblob

In [0]:
import lseg.data as ld
import numpy as np
import pandas as pd

from datetime import datetime, timedelta

from bs4 import BeautifulSoup
from textblob import TextBlob
import datetime
from datetime import time
import warnings
warnings.filterwarnings("ignore")
import configparser as cp
#https://github.com/Refinitiv-API-Samples/Article.EikonAPI.Python.NewsSentimentAnalysis

In [0]:
# Fetch Apple stock prices for the last year using ld API
# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")

In [0]:
import pandas as pd
pd.merge


In [0]:
from datetime import datetime, timedelta
end = datetime.today()
start = end - timedelta(days=(365.2*4))
print(start, end)

In [0]:
import requests

# Finnhub free API key — stored in Databricks Secrets
FINNHUB_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="finnhub_key")

# Format dates for Finnhub API (YYYY-MM-DD)
from_str = start.strftime('%Y-%m-%d')
to_str = end.strftime('%Y-%m-%d')

# Fetch company news from Finnhub (free tier, up to 1000 articles per call)
url = "https://finnhub.io/api/v1/company-news"
params = {
    "symbol": "MSFT",
    "from": from_str,
    "to": to_str,
    "token": FINNHUB_KEY
}
response = requests.get(url, params=params)
response.raise_for_status()
news_data = response.json()

# Convert to DataFrame with column names compatible with downstream cells
df = pd.DataFrame(news_data)
df['versionCreated'] = pd.to_datetime(df['datetime'], unit='s')
df['text'] = df['headline']          # headline text (maps to original 'text' column)
df['storyId'] = df['id'].astype(str)  # unique story ID
df = df[['storyId', 'versionCreated', 'text', 'headline', 'summary', 'source', 'url']]
print(f"Fetched {len(df)} MSFT news articles from {from_str} to {to_str}")
df.head()

In [0]:
# Fetch sample news story text from Finnhub (summary field already contains article text)
newsText = df['summary'].iloc[0]
newsText

In [0]:
soup = BeautifulSoup(newsText,"lxml")
sentA = TextBlob(soup.get_text())
sentA.sentiment.polarity

In [0]:
# Initialize sentiment columns
df['Polarity'] = np.nan
df['Subjectivity'] = np.nan
df['Score'] = None

for idx, storyId in enumerate(df['storyId'].values):  # for each row in our df dataframe
    newsText = df['summary'].iloc[idx]  # get news text from Finnhub summary field
    if newsText:
        soup = BeautifulSoup(newsText, "lxml")  # create a BeautifulSoup object from our HTML news article
        sentA = TextBlob(soup.get_text())  # pass the text only article to TextBlob to analyse
        df.loc[idx, 'Polarity'] = sentA.sentiment.polarity  # write sentiment polarity back to df
        df.loc[idx, 'Subjectivity'] = sentA.sentiment.subjectivity  # write sentiment subjectivity score back to df
        if sentA.sentiment.polarity >= 0.05:  # attribute bucket to sentiment polarity
            score = 'positive'
        elif -.05 < sentA.sentiment.polarity < 0.05:
            score = 'neutral'
        else:
            score = 'negative'
        df.loc[idx, 'Score'] = score  # write score back to df
df.head()

In [0]:
start = df['versionCreated'].min().replace(hour=0, minute=0, second=0, microsecond=0)
end = df['versionCreated'].max().replace(hour=0, minute=0, second=0, microsecond=0)
# Use LSEG field names (equivalent to Eikon's OPEN, HIGH, LOW, CLOSE, VOLUME)
Minute = ld.get_history(
    universe="MSFT.O",
    fields=["OPEN_PRC", "HIGH_1", "LOW_1", "TRDPRC_1", "ACVOL_UNS"],
    start=start, end=end, interval="1min"
)
# Flatten MultiIndex columns if present
if isinstance(Minute.columns, pd.MultiIndex):
    Minute.columns = Minute.columns.get_level_values(-1)
# Ensure index is timezone-naive for compatibility with downstream cells
if Minute.index.tz is not None:
    Minute.index = Minute.index.tz_localize(None)
print(f"Minute data: {Minute.index.min()} to {Minute.index.max()} ({len(Minute)} bars)")
Minute

In [0]:
# Analyze the effect of news on price after 1, 5, 10, and 60 minutes
# Uses df (news with sentiment + versionCreated) and Minute (1-min OHLCV bars)

windows = [1, 5, 10, 60]  # minutes after news

# Work on a copy so we don't mutate the originals
news_df = df[['storyId', 'versionCreated', 'Score', 'Polarity']].copy()

# Ensure Minute index is sorted and tz-naive (done in prior cell, but be safe)
Minute = Minute.sort_index()
if Minute.index.tz is not None:
    Minute.index = Minute.index.tz_localize(None)

# Use TRDPRC_1 as the close price column
price_col = 'TRDPRC_1' if 'TRDPRC_1' in Minute.columns else Minute.columns[-1]

results = []

for _, row in news_df.iterrows():
    news_time = row['versionCreated']
    if pd.isna(news_time):
        continue
    # Floor to the nearest available minute bar
    idx_pos = Minute.index.searchsorted(news_time, side='left')
    if idx_pos >= len(Minute):
        continue  # news after last available bar
    
    base_price = Minute.iloc[idx_pos][price_col]
    if pd.isna(base_price):
        continue
    
    record = {
        'storyId': row['storyId'],
        'news_time': news_time,
        'Score': row['Score'],
        'Polarity': row['Polarity'],
        'base_price': base_price,
    }
    
    for w in windows:
        target_time = news_time + pd.Timedelta(minutes=w)
        target_pos = Minute.index.searchsorted(target_time, side='left')
        if target_pos < len(Minute):
            future_price = Minute.iloc[target_pos][price_col]
            if pd.notna(future_price):
                record[f'price_t{w}'] = future_price
                record[f'pct_chg_t{w}'] = (future_price - base_price) / base_price * 100
            else:
                record[f'price_t{w}'] = np.nan
                record[f'pct_chg_t{w}'] = np.nan
        else:
            record[f'price_t{w}'] = np.nan
            record[f'pct_chg_t{w}'] = np.nan
    
    results.append(record)

price_effect_df = pd.DataFrame(results)

# Summary: average % price change by sentiment score and time window
summary = price_effect_df.groupby('Score')[
    [f'pct_chg_t{w}' for w in windows]
].mean().round(4)
summary.columns = [f'{w} min' for w in windows]

print("=== Average % price change after news by sentiment ===")
print(summary)
print()
print(f"\nTotal news events matched to price data: {len(price_effect_df)}")
print(f"\nFull detail (first 10 rows):")
price_effect_df.head(10)